In [12]:
from sklearn.preprocessing import OneHotEncoder
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
import pandas as pd
from joblib import dump
import os

In [13]:
df = pd.read_csv(r"C:\Users\pc\Fraud-Detection\data\fraud.csv")

In [14]:
X = df.drop(columns=["class", "ip_address", "signup_time", "purchase_time", "device_id"])
y = df["class"]

# 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [15]:
# Step 1: Identify categorical columns
categorical_cols = X_train.select_dtypes(include='object').columns

# Step 2: Fit OneHotEncoder on training categorical columns
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(X_train[categorical_cols])

# Step 3: Transform both X_train and X_test
X_train_encoded = pd.DataFrame(
    encoder.transform(X_train[categorical_cols]),
    columns=encoder.get_feature_names_out(categorical_cols),
    index=X_train.index
)

X_test_encoded = pd.DataFrame(
    encoder.transform(X_test[categorical_cols]),
    columns=encoder.get_feature_names_out(categorical_cols),
    index=X_test.index
)

# Step 4: Drop original categorical columns and combine encoded with numeric
X_train_final = pd.concat([X_train.drop(columns=categorical_cols), X_train_encoded], axis=1)
X_test_final = pd.concat([X_test.drop(columns=categorical_cols), X_test_encoded], axis=1)

# Step 5: Apply SMOTE only on training set
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_final, y_train)


In [16]:


# Create the directory if it doesn't exist
os.makedirs("artifacts", exist_ok=True)

# Save your processed datasets
dump(X_train_bal, "artifacts/X_train_bal.joblib")
dump(y_train_bal, "artifacts/y_train_bal.joblib")
dump(X_test_final, "artifacts/X_test_final.joblib")
dump(y_test, "artifacts/y_test.joblib")


['artifacts/y_test.joblib']